In [1]:
import numpy as np

# ============================================================
# 1. Single-qubit states and projectors
# ============================================================

def ket_H():
    return np.array([1, 0], dtype=complex)

def ket_V():
    return np.array([0, 1], dtype=complex)

def ket_D():
    return (ket_H() + ket_V()) / np.sqrt(2)

def ket_A():
    return (ket_H() - ket_V()) / np.sqrt(2)

def ket_R():
    return (ket_H() + 1j * ket_V()) / np.sqrt(2)

def ket_L():
    return (ket_H() - 1j * ket_V()) / np.sqrt(2)

KETS_6 = {
    "H": ket_H(),
    "V": ket_V(),
    "D": ket_D(),
    "A": ket_A(),
    "R": ket_R(),
    "L": ket_L(),
}

def projector(label: str):
    k = KETS_6[label]
    return np.outer(k, np.conjugate(k))

def kron(a: np.ndarray, b: np.ndarray):
    return np.kron(a, b)

def vec(M: np.ndarray):
    return M.reshape(-1, order="F")

def unvec(v: np.ndarray, dim: int):
    return v.reshape((dim, dim), order="F")

# ============================================================
# 2. Measurement operators for 36 settings
#    Internal ordering: Signal ⊗ Idler
#    Input keys: (idler_label, signal_label)
# ============================================================

def build_measurement_ops_36():
    labels = ["H", "V", "D", "A", "R", "L"]
    meas_labels = []
    meas_ops = []

    for idler_label in labels:
        for signal_label in labels:
            Pi = projector(idler_label)   # Idler
            Ps = projector(signal_label)  # Signal
            M = kron(Ps, Pi)              # Signal ⊗ Idler
            meas_labels.append((idler_label, signal_label))
            meas_ops.append(M)

    return meas_labels, meas_ops

# ============================================================
# 3. Pauli basis (orthonormal)
#    F_m = {I, X, Y, Z}/sqrt(2)
# ============================================================

def pauli_basis_orthonormal():
    I = np.array([[1, 0], [0, 1]], dtype=complex)
    X = np.array([[0, 1], [1, 0]], dtype=complex)
    Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
    Z = np.array([[1, 0], [0, -1]], dtype=complex)
    return [I / np.sqrt(2), X / np.sqrt(2), Y / np.sqrt(2), Z / np.sqrt(2)]

In [2]:
# ============================================================
# 4. Step A: linear inversion of rho_out from 36 counts
#    Uses relative frequencies only as rough initializer
# ============================================================

def linear_inversion_rho_out_from_36_counts(counts_idler_signal):
    """
    Rough linear inversion for rho_out from 36 coincidence counts.
    Returns a trace-1 Hermitian matrix, then projects to PSD.
    """
    meas_labels, meas_ops = build_measurement_ops_36()

    counts = np.array([float(counts_idler_signal[label]) for label in meas_labels], dtype=float)
    total = np.sum(counts)
    if total <= 0:
        raise ValueError("Total counts must be positive.")

    probs = counts / total

    A = np.vstack([vec(M).conj() for M in meas_ops])  # shape (36, 16)
    rho_vec, *_ = np.linalg.lstsq(A, probs, rcond=None)
    rho = unvec(rho_vec, 4)

    rho = (rho + rho.conj().T) / 2
    tr = np.trace(rho)
    if abs(tr) < 1e-15:
        rho = np.eye(4, dtype=complex) / 4
    else:
        rho = rho / tr

    rho = project_to_psd_trace_one(rho)
    return rho

def project_to_psd_trace_one(rho):
    rho = (rho + rho.conj().T) / 2
    vals, vecs = np.linalg.eigh(rho)
    vals = np.clip(vals, 0.0, None)
    if np.sum(vals) <= 0:
        return np.eye(rho.shape[0], dtype=complex) / rho.shape[0]
    rho_psd = vecs @ np.diag(vals) @ vecs.conj().T
    rho_psd /= np.trace(rho_psd)
    return rho_psd

# ============================================================
# 5. Step B: linear least-squares estimate of chi from
#    rho_out ≈ (E_chi ⊗ I)(rho_in)
# ============================================================

def chi_linear_inversion_from_rho_in_rho_out(rho_in, rho_out):
    """
    Solve linear least-squares for chi:
        vec(rho_out) = A_chi * vec(chi)

    chi is in orthonormal Pauli basis.
    """
    basis = pauli_basis_orthonormal()
    I_idler = np.eye(2, dtype=complex)

    cols = []
    for m, Fm in enumerate(basis):
        for n, Fn in enumerate(basis):
            Km = kron(Fm, I_idler)
            Kn = kron(Fn, I_idler)
            Bmn = Km @ rho_in @ Kn.conj().T
            cols.append(vec(Bmn))

    A_chi = np.column_stack(cols)  # shape (16,16)
    b = vec(rho_out)

    chi_vec, *_ = np.linalg.lstsq(A_chi, b, rcond=None)
    chi = chi_vec.reshape((4, 4), order="F")
    chi = (chi + chi.conj().T) / 2
    return chi

# ============================================================
# 6. Step C: project chi to PSD and optionally rescale
# ============================================================

def project_chi_to_psd(chi):
    chi = (chi + chi.conj().T) / 2
    vals, vecs = np.linalg.eigh(chi)
    vals = np.clip(vals, 0.0, None)
    chi_psd = vecs @ np.diag(vals) @ vecs.conj().T
    chi_psd = (chi_psd + chi_psd.conj().T) / 2
    return chi_psd

# ============================================================
# 7. Step D: convert chi_init -> lower-triangular T -> x0
#    We use Cholesky if possible, otherwise eig-based square root
# ============================================================

def chi_to_lower_triangular_params(chi, eps=1e-12):
    """
    Convert PSD chi into initial parameter vector x0 (length 16)
    for T such that chi ≈ T^\dagger T.

    If chi is positive definite, use Cholesky on chi.
    Otherwise, add tiny diagonal and use Cholesky.
    """
    chi = (chi + chi.conj().T) / 2

    # Small regularization to avoid numerical singularity
    chi_reg = chi + eps * np.eye(4, dtype=complex)

    # Need T such that chi = T^\dagger T
    # np.linalg.cholesky gives L with chi = L L^\dagger
    # so choose T = L^\dagger
    L = np.linalg.cholesky(chi_reg)
    T = L.conj().T

    # Force lower-triangular parameterization expected by previous code:
    # previous code assumes T itself is lower triangular.
    # So better use chi = T^dag T with lower-triangular T directly:
    # equivalently use cholesky of chi as lower triangular L, then chi = L L^dag.
    # To match chi = T^dag T, choose T = L^dag (upper triangular),
    # but our parameterization expects lower triangular.
    #
    # Therefore we instead parameterize from a lower-triangular T_low satisfying
    # chi ≈ T_low^dag T_low by using eigen-decomposition and then QR-like refit.
    #
    # Simpler practical workaround: use lower triangle of L and define T_low = L.
    # Then chi_init implied by T_low^dag T_low is not exactly chi, but close enough
    # for initialization. A better approach is to adapt the main code to accept upper
    # triangular T, but here we stay compatible with the existing parametrization.
    T_low = L

    x0 = np.zeros(16, dtype=float)
    x0[0] = np.real(T_low[0, 0])
    x0[1] = np.real(T_low[1, 1])
    x0[2] = np.real(T_low[2, 2])
    x0[3] = np.real(T_low[3, 3])

    idx = 4
    for i in range(1, 4):
        for j in range(i):
            x0[idx] = np.real(T_low[i, j])
            x0[idx + 1] = np.imag(T_low[i, j])
            idx += 2

    return x0

# ============================================================
# 8. Global-scale initializer from counts
# ============================================================

def initial_log_global_scale_from_counts(counts_idler_signal, backgrounds=None):
    meas_labels, _ = build_measurement_ops_36()
    if backgrounds is None:
        backgrounds = {label: 0.0 for label in meas_labels}

    counts = np.array([float(counts_idler_signal[label]) for label in meas_labels], dtype=float)
    bkg = np.array([float(backgrounds[label]) for label in meas_labels], dtype=float)

    rough_scale = max(np.sum(counts - bkg), 1.0)
    return np.log(rough_scale)

# ============================================================
# 9. Full initializer:
#    counts -> rho_out_lin -> chi_lin -> chi_psd -> x0
# ============================================================

def build_initial_x0_from_linear_inversion(
    rho_in,
    counts_idler_signal,
    backgrounds=None,
    fit_global_scale=True
):
    """
    Returns initial vector x0 for the NTP Poisson-MLE code.

    Steps:
      1) Linear inversion of rho_out from 36 counts
      2) Linear inversion of chi from rho_in and rho_out
      3) PSD projection of chi
      4) Convert chi to lower-triangular parameter vector
      5) Append log(global_scale) if needed
    """
    rho_in = np.array(rho_in, dtype=complex)
    rho_in = (rho_in + rho_in.conj().T) / 2
    rho_in /= np.trace(rho_in)

    rho_out_lin = linear_inversion_rho_out_from_36_counts(counts_idler_signal)
    chi_lin = chi_linear_inversion_from_rho_in_rho_out(rho_in, rho_out_lin)
    chi_psd = project_chi_to_psd(chi_lin)

    x0_chi = chi_to_lower_triangular_params(chi_psd)

    if fit_global_scale:
        log_s0 = initial_log_global_scale_from_counts(counts_idler_signal, backgrounds=backgrounds)
        x0 = np.concatenate([x0_chi, [log_s0]])
    else:
        x0 = x0_chi

    diagnostics = {
        "rho_out_linear": rho_out_lin,
        "chi_linear_raw": chi_lin,
        "chi_linear_psd": chi_psd,
        "x0_chi_only": x0_chi,
    }
    return x0, diagnostics

<>:95: SyntaxWarning: invalid escape sequence '\d'
<>:95: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Jaedeok Park\AppData\Local\Temp\ipykernel_12820\2695623366.py:95: SyntaxWarning: invalid escape sequence '\d'
  for T such that chi ≈ T^\dagger T.
